# 🎓 The Dropout Signal — Dashboard
## Visual Analytics for Student Retention | HackBricks 2026

**Prerequisite:** Run `dropout_pipeline.ipynb` → `dropout_ml.ipynb` first.

**Panels:**
1. KPI Summary
2. Risk Tier Distribution (Pie)
3. Top Risk Factors (Bar)
4. Grade Trajectory by Outcome (Box Plot)
5. Financial Stress vs Risk Score (Scatter)
6. Fairness Audit Results
7. High-Risk Student Alert Feed
8. Confusion Matrix

In [ ]:
# ============================================================================
#  Config
# ============================================================================
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = "default"

BRONZE_TABLE   = f"{CATALOG}.{SCHEMA}.dropout_bronze"
SILVER_TABLE   = f"{CATALOG}.{SCHEMA}.dropout_silver"
GOLD_TABLE     = f"{CATALOG}.{SCHEMA}.dropout_gold"
FAIRNESS_TABLE = f"{CATALOG}.{SCHEMA}.dropout_fairness_audit"

print("✅ Dashboard ready — run all cells below")

---
## 📊 Panel 1 — KPI Summary

In [ ]:
# ============================================================================
#  Panel 1: KPI Summary
# ============================================================================
gold = spark.table(GOLD_TABLE)
silver = spark.table(SILVER_TABLE)

kpi = gold.agg(
    F.count("*").alias("total_students"),
    F.sum(F.when(F.col("risk_tier") == "HIGH", 1).otherwise(0)).alias("high_risk"),
    F.sum(F.when(F.col("risk_tier") == "MEDIUM", 1).otherwise(0)).alias("medium_risk"),
    F.sum(F.when(F.col("risk_tier") == "LOW", 1).otherwise(0)).alias("low_risk"),
    F.round(F.avg("risk_score") * 100, 1).alias("avg_risk_pct"),
    F.sum(F.when(F.col("predicted_dropout") == 1, 1).otherwise(0)).alias("predicted_dropouts"),
).collect()[0]

print("=" * 60)
print("  🎓 The Dropout Signal — KPIs")
print("=" * 60)
print(f"  📊 Total Students:       {kpi['total_students']:,}")
print(f"  🔴 HIGH Risk:            {kpi['high_risk']:,}")
print(f"  🟡 MEDIUM Risk:          {kpi['medium_risk']:,}")
print(f"  🟢 LOW Risk:             {kpi['low_risk']:,}")
print(f"  📈 Avg Risk Score:       {kpi['avg_risk_pct']}%")
print(f"  ⚠️  Predicted Dropouts:   {kpi['predicted_dropouts']:,}")
print("=" * 60)

---
## 🥧 Panel 2 — Risk Tier Distribution
Select **Pie Chart** → Keys: `risk_tier`, Values: `count`

In [ ]:
# ============================================================================
#  Panel 2: Risk Tier Distribution (Pie Chart)
# ============================================================================
display(
    spark.sql(f"""
        SELECT risk_tier, COUNT(*) AS count,
               ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM {GOLD_TABLE}), 1) AS pct
        FROM {GOLD_TABLE}
        GROUP BY risk_tier
        ORDER BY CASE risk_tier WHEN 'HIGH' THEN 1 WHEN 'MEDIUM' THEN 2 ELSE 3 END
    """)
)

---
## 📉 Panel 3 — Grade Trajectory: Dropouts vs Graduates
Do dropouts show declining grades? Select **Bar Chart** → Keys: `outcome`, Values: `avg_grade_trajectory`

In [ ]:
# ============================================================================
#  Panel 3: Grade Trajectory by Outcome
# ============================================================================
display(
    spark.sql(f"""
        SELECT
            target AS outcome,
            ROUND(AVG(grade_trajectory), 3)     AS avg_grade_trajectory,
            ROUND(AVG(avg_grade_sem1), 2)       AS avg_sem1_grade,
            ROUND(AVG(avg_grade_sem2), 2)       AS avg_sem2_grade,
            ROUND(AVG(financial_stress_index), 3) AS avg_financial_stress,
            COUNT(*) AS count
        FROM {SILVER_TABLE}
        GROUP BY target
        ORDER BY target
    """)
)

---
## ⚖️ Panel 4 — Fairness Audit Results
The key table that proves our model is fair (or documents where it isn't).

In [ ]:
# ============================================================================
#  Panel 4: Fairness Audit Results
# ============================================================================
display(
    spark.sql(f"""
        SELECT
            protected_attribute,
            group_a, group_b,
            group_a_count, group_b_count,
            demographic_parity_ratio, dp_is_fair,
            equal_opportunity_ratio, eo_is_fair,
            predictive_parity_ratio, pp_is_fair
        FROM {FAIRNESS_TABLE}
    """)
)

---
## 📊 Panel 5 — Engineered Features: Dropouts vs Graduates
How do our custom features separate the two groups? Select **Bar Chart** → Keys: `outcome`

In [ ]:
# ============================================================================
#  Panel 5: Engineered Feature Comparison
# ============================================================================
display(
    spark.sql(f"""
        SELECT
            target AS outcome,
            ROUND(AVG(grade_trajectory), 3)            AS avg_grade_trajectory,
            ROUND(AVG(financial_stress_index), 3)      AS avg_financial_stress,
            ROUND(AVG(curricular_engagement_sem1), 3)  AS avg_engagement_sem1,
            ROUND(AVG(curricular_engagement_sem2), 3)  AS avg_engagement_sem2,
            ROUND(AVG(economic_pressure_index), 3)     AS avg_economic_pressure,
            ROUND(AVG(age_risk_flag) * 100, 1)         AS pct_mature_students
        FROM {SILVER_TABLE}
        GROUP BY target
        ORDER BY target
    """)
)

---
## 🔎 Panel 6 — Risk Score Distribution
Select **Histogram** or **Bar Chart** → Keys: `risk_bucket`, Values: `count`

In [ ]:
# ============================================================================
#  Panel 6: Risk Score Distribution
# ============================================================================
display(
    spark.sql(f"""
        SELECT
            CASE
                WHEN risk_score >= 0.9 THEN '0.9-1.0'
                WHEN risk_score >= 0.8 THEN '0.8-0.9'
                WHEN risk_score >= 0.7 THEN '0.7-0.8'
                WHEN risk_score >= 0.6 THEN '0.6-0.7'
                WHEN risk_score >= 0.5 THEN '0.5-0.6'
                WHEN risk_score >= 0.4 THEN '0.4-0.5'
                WHEN risk_score >= 0.3 THEN '0.3-0.4'
                WHEN risk_score >= 0.2 THEN '0.2-0.3'
                WHEN risk_score >= 0.1 THEN '0.1-0.2'
                ELSE '0.0-0.1'
            END AS risk_bucket,
            COUNT(*) AS count
        FROM {GOLD_TABLE}
        GROUP BY risk_bucket
        ORDER BY risk_bucket
    """)
)

## 🚨 Panel 7 — High-Risk Student Alert Feed
> Top 20 highest-risk students with their risk factors and recommended interventions.
> **Action**: This is the counselor's working list — sorted by urgency.

In [ ]:
# ============================================================================
#  Panel 7: High-Risk Student Alert Feed
# ============================================================================
high_risk_df = spark.sql(f"""
    SELECT
        student_id,
        ROUND(risk_score, 3) AS risk_score,
        risk_tier,
        top_factor_1,
        top_factor_2,
        top_factor_3,
        recommended_action
    FROM {GOLD_TABLE}
    WHERE risk_tier = 'HIGH'
    ORDER BY risk_score DESC
    LIMIT 20
""")

print(f"🚨 HIGH-RISK STUDENTS — Top 20 (of {spark.sql(f\"SELECT COUNT(*) FROM {GOLD_TABLE} WHERE risk_tier = 'HIGH'\").collect()[0][0]} total)")
print("=" * 120)
display(high_risk_df)

## 🎯 Panel 8 — Prediction Accuracy: Confusion Matrix
> Cross-tabulation of **predicted vs actual** outcomes from the Gold table.
> Shows how well the champion model separates real dropouts from continuing students.

In [ ]:
# ============================================================================
#  Panel 8: Confusion Matrix — Predicted vs Actual
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np

gold_pdf = spark.sql(f"""
    SELECT predicted_dropout, actual_outcome
    FROM {GOLD_TABLE}
""").toPandas()

# Build confusion matrix manually
tp = ((gold_pdf["predicted_dropout"] == 1) & (gold_pdf["actual_outcome"] == 1)).sum()
tn = ((gold_pdf["predicted_dropout"] == 0) & (gold_pdf["actual_outcome"] == 0)).sum()
fp = ((gold_pdf["predicted_dropout"] == 1) & (gold_pdf["actual_outcome"] == 0)).sum()
fn = ((gold_pdf["predicted_dropout"] == 0) & (gold_pdf["actual_outcome"] == 1)).sum()

cm = np.array([[tn, fp], [fn, tp]])
total = cm.sum()
accuracy = (tp + tn) / total * 100

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted: Stay", "Predicted: Dropout"], fontsize=11)
ax.set_yticklabels(["Actual: Stay", "Actual: Dropout"], fontsize=11)

# Annotate cells
for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, f"{cm[i, j]}\n({cm[i, j]/total*100:.1f}%)",
                ha="center", va="center", fontsize=14, fontweight="bold", color=color)

ax.set_title(f"Confusion Matrix — Overall Accuracy: {accuracy:.1f}%", fontsize=14, fontweight="bold")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print(f"\n📊 True Positives (caught dropouts): {tp}")
print(f"📊 True Negatives (correctly passed): {tn}")
print(f"📊 False Positives (false alarms): {fp}")
print(f"📊 False Negatives (missed dropouts): {fn}")
print(f"✅ Overall Accuracy: {accuracy:.1f}%")

---

# ✅ Dashboard Complete — The Dropout Signal

| Panel | What it shows |
|-------|--------------|
| 1 — KPI Summary | Total students, risk tier counts, average risk score |
| 2 — Risk Tier Pie | Visual split: HIGH / MEDIUM / LOW |
| 3 — Grade Trajectory | Semester 1 → 2 grade change by outcome |
| 4 — Fairness Audit | DPR, EOR, PP across gender & socioeconomic groups |
| 5 — Engineered Features | All 6 custom features compared by dropout status |
| 6 — Risk Distribution | Risk score histogram in 10 buckets |
| 7 — Alert Feed | Top 20 highest-risk students with factors & actions |
| 8 — Confusion Matrix | Predicted vs actual with accuracy metrics |

> **Judges**: Every panel tells a story — from the big picture (Panels 1-2) through fairness (Panel 4) to individual student intervention (Panel 7).

---
*HackBricks 2026 — The Dropout Signal*